# Actual-Run Baseline Comparison

This notebook scores the reusable baselines from `calculate_baselines.py` against the exact evaluation cells from completed elicitation runs.

Workflow:

1. Configure `RUN_PATHS` with run directories or `*_estimates.csv` files.
2. Load final valid elicitation rows from every run.
3. Check early that all runs used compatible target tasks and evaluation cells.
4. Build baseline predictions once on the shared evaluation panel.
5. Produce one comparison table with Brier and Beta-CRPS metrics plus row-bootstrap confidence intervals for every baseline and every included run.

The target-task and evaluation-cell checks are intentionally before the Lyptus/IRT baseline setup. Target tasks that are not present in every run are reported and excluded from the shared scoring panel.


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
pd.set_option("display.max_colwidth", 140)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "intra_benchmark_calibration").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing intra_benchmark_calibration")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from report_analyses.calculate_baselines import (
    DEFAULT_ORDERED_SCORE_COLUMNS,
    BaselineConfig,
    build_baseline_context,
    calculate_baselines,
    load_shared_run_comparison,
    score_runs,
)

REPO_ROOT


## Configuration

Edit `RUN_PATHS` to point at the completed runs you want to compare. Each entry can be either a run directory containing a `*_estimates.csv` file or the CSV itself.

`STRICT_EVALUATION_CELL_MATCH` is separate from the target-task intersection. Keeping it `True` warns when scoreable evaluation cells differ before the notebook falls back to the cells present in every run.

In [ ]:
DATA_PATH = REPO_ROOT / "intra_benchmark_calibration" / "experiments" / "G_model_sweep" / "results"
RUN_PATHS = [
    # Replace or extend this list with run directories or direct *_estimates.csv paths.
    DATA_PATH / "gemini25flash",
    DATA_PATH / "gpt55",
    DATA_PATH / "haiku45",
    DATA_PATH / "opus47",
    DATA_PATH / "sonnet46",
]

# Optional labels keyed by the exact Path object in RUN_PATHS. Any omitted path gets a label from its directory/file name.
RUN_LABELS: dict[Path, str] = {}

# Use "all" or any subset of: uninformed_0_5, model_pass_rate, model_bin_pass_rate, irt_logistic_fit.
BASELINES = "all"

STRICT_EVALUATION_CELL_MATCH = True
SCORE_ONE_ROW_PER_EVALUATION_CELL = False

LYPTUS_REPO_DIR = Path("/home/jeffm/lyptus-data")
DROP_MODELS = ("GPT-2", "GPT-3", "GPT-3.5")

N_BINS = 5
BINNING_STRATEGY = "equal_count"
EXPLICIT_EDGES = None

BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_SEED = 20260515
METRIC_BOOTSTRAP_QUANTILES = (0.025, 0.975)

IRT_BOOTSTRAP_ITERATIONS = 250
IRT_BOOTSTRAP_SEED = BOOTSTRAP_SEED + 1000

# Set to None to disable saving tables and figures.
RESULTS_DIR: Path | None = REPO_ROOT / "report_analyses" / "results" / "model_sweep_baseline"
COMPARISON_TABLE_CSV = "comparison_table.csv"
BASELINE_SCORES_CSV = "baseline_scores.csv"
RUN_SCORES_CSV = "run_scores.csv"
SCORE_FIGURE_PNG = "score_comparison.png"
SCORE_FIGURE_PDF = "score_comparison.pdf"

BASELINE_CONFIG = BaselineConfig(
    lyptus_repo_dir=LYPTUS_REPO_DIR,
    drop_models=DROP_MODELS,
    n_bins=N_BINS,
    binning_strategy=BINNING_STRATEGY,
    explicit_edges=EXPLICIT_EDGES,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    bootstrap_seed=BOOTSTRAP_SEED,
    metric_bootstrap_quantiles=METRIC_BOOTSTRAP_QUANTILES,
    irt_bootstrap_iterations=IRT_BOOTSTRAP_ITERATIONS,
    irt_bootstrap_seed=IRT_BOOTSTRAP_SEED,
)


## Load Runs And Check Shared Targets

`load_shared_run_comparison` extracts the target tasks from every run, reports target/task-cell mismatches, and creates the shared scoring panel used below.


In [ ]:
run_data = load_shared_run_comparison(
    RUN_PATHS,
    run_labels=RUN_LABELS,
    strict_evaluation_cell_match=STRICT_EVALUATION_CELL_MATCH,
)

raw_run_frames = run_data.raw_run_frames
run_frames = run_data.run_frames
shared_run_frames = run_data.shared_run_frames
shared_panel = run_data.shared_panel

display(run_data.run_index)
display(run_data.target_count_summary)

if len(run_data.incomplete_targets):
    display(run_data.incomplete_targets.sort_values(["n_runs", "target_task_id"]).reset_index(drop=True))

if len(run_data.cell_mismatches):
    display(run_data.cell_mismatches)

print(
    f"Shared raw target-task panel has {len(run_data.common_target_ids)} "
    f"target tasks present in all {len(run_frames)} runs."
)
print(f"Shared scoreable evaluation panel has {len(shared_panel)} forecasted-model/task cells.")


## Load Lyptus Outcomes, Bins, And IRT Fits

The baseline machinery below mirrors `baseline_comparator_analysis.ipynb`. These baseline predictions are built once on `shared_panel`, which defaults to tasks present in every run.

In [ ]:
baseline_context = build_baseline_context(BASELINE_CONFIG, baselines=BASELINES)

dataset = baseline_context.dataset
bins = baseline_context.bins
tasks = baseline_context.tasks
irt_fit_data = baseline_context.irt_fit_data
irt_fit_params = baseline_context.irt_fit_params
irt_bootstrap_fit_params = baseline_context.irt_bootstrap_fit_params

if not irt_bootstrap_fit_params.empty:
    display(
        irt_bootstrap_fit_params.groupby("agent")
        .agg(n_bootstrap=("bootstrap_idx", "size"), success_rate=("fit_success", "mean"))
        .loc[[model for model in dataset.outcomes.models if model in irt_bootstrap_fit_params["agent"].unique()]]
    )


## Build Baseline Predictions Once

`uninformed_0_5` keeps the original point prediction of `0.5` for Brier. For Beta CRPS, this notebook gives it an explicitly uninformed `Beta(1, 1)` distribution, whose quartiles are `0.25, 0.5, 0.75`, so every row in the final table has a CRPS entry.

In [ ]:
baseline_scores, panel_with_baselines = calculate_baselines(
    panel=shared_panel,
    context=baseline_context,
    baselines=BASELINES,
    return_panel=True,
    source_columns=True,
)
display(panel_with_baselines.head())


## Score Baselines And Runs

Brier is scored on each source's point prediction (`p50` for elicited runs). Beta CRPS is computed from `p25/p50/p75` using the same helper as `analyse_results.py`. Confidence intervals are non-parametric row bootstraps over the scored rows.

In [ ]:
run_scores = score_runs(
    shared_run_frames,
    config=BASELINE_CONFIG,
    one_row_per_evaluation_cell=SCORE_ONE_ROW_PER_EVALUATION_CELL,
)

comparison_table = pd.concat([baseline_scores, run_scores], ignore_index=True)
comparison_table = comparison_table.sort_values("brier", kind="stable").reset_index(drop=True)
comparison_table = comparison_table[DEFAULT_ORDERED_SCORE_COLUMNS]

if RESULTS_DIR is not None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    table_outputs = {
        COMPARISON_TABLE_CSV: comparison_table,
        BASELINE_SCORES_CSV: baseline_scores,
        RUN_SCORES_CSV: run_scores,
    }
    for filename, table in table_outputs.items():
        output_path = RESULTS_DIR / filename
        table.to_csv(output_path, index=False)
        print(f"Wrote {output_path}")

display(comparison_table)


In [ ]:
import matplotlib.pyplot as plt


def plot_metric_bars(ax, table: pd.DataFrame, metric: str, ci_low: str, ci_high: str, title: str) -> None:
    plot_data = table.dropna(subset=[metric]).sort_values(metric, kind="stable").reset_index(drop=True)
    labels = plot_data["source"].astype(str)
    values = plot_data[metric]
    lower_errors = values - plot_data[ci_low]
    upper_errors = plot_data[ci_high] - values
    colors = plot_data["source_type"].map({"baseline": "#9ca3af"}).fillna("#4682b4")

    ax.bar(
        labels,
        values,
        yerr=[lower_errors, upper_errors],
        color=colors,
        edgecolor="white",
        linewidth=0.8,
        capsize=3,
        error_kw={"elinewidth": 1, "ecolor": "#374151"},
    )
    ax.set_title(title)
    ax.set_ylabel("Score; lower is better")
    ax.tick_params(axis="x", labelrotation=35)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
    ax.grid(axis="y", alpha=0.25)
    ax.set_axisbelow(True)


fig, axes = plt.subplots(ncols=2, figsize=(14, 5.5), constrained_layout=True)
plot_metric_bars(axes[0], comparison_table, "brier", "brier_ci_low", "brier_ci_high", "Brier Score")
plot_metric_bars(axes[1], comparison_table, "crps_beta", "crps_beta_ci_low", "crps_beta_ci_high", "Beta CRPS")

legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color="#9ca3af", label="Baseline"),
    plt.Rectangle((0, 0), 1, 1, color="#4682b4", label="Model run"),
]
fig.legend(handles=legend_handles, loc="lower center", ncol=2, frameon=False)

if RESULTS_DIR is not None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    for filename in [SCORE_FIGURE_PNG, SCORE_FIGURE_PDF]:
        output_path = RESULTS_DIR / filename
        fig.savefig(output_path, bbox_inches="tight", dpi=300)
        print(f"Wrote {output_path}")

plt.show()
